In [4]:
import pandas as pd
import numpy as np
import geopandas as gpd
import rasterio
import matplotlib.pyplot as plt

from pathlib import Path

In [6]:
PROJECT_ROOT = Path.cwd().parent.parent

DATA_DIR = PROJECT_ROOT / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
PROCESSED_DATA_DIR = DATA_DIR / "processed"

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)


In [7]:
# Load our Population Csv 
population_file = RAW_DATA_DIR / "ghana_population_by_geographic_area_2021.csv"

population_df = pd.read_csv(population_file)

population_df.head()

,region,population
0,Ghana,30832019
1,Western,2060585
2,Jomoro Municipal,126576
3,Ellembelle,120893
4,Nzema East Municipal,94621


In [ ]:
population_df.shape

In [10]:
population_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 295 entries, 0 to 294
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   region      295 non-null    str  
 1   population  295 non-null    int64
dtypes: int64(1), str(1)
memory usage: 4.7 KB


In [11]:
population_df.describe(include="all")

,region,population
count,295,2.950000e+02
unique,295,NaN
top,Ghana,NaN
freq,1,NaN
mean,NaN,3.193632e+05
std,NaN,1.864413e+06
min,NaN,2.817900e+04
25%,NaN,7.439550e+04
50%,NaN,1.063740e+05
75%,NaN,1.533150e+05


In [12]:
population_df.isnull().sum()

region        0
population    0
dtype: int64

In [13]:
population_df.duplicated().sum()

np.int64(0)

In [14]:
population_df["region"].unique()

<StringArray>
[                                    'Ghana',
                                   'Western',
                          'Jomoro Municipal',
                                'Ellembelle',
                      'Nzema East Municipal',
                     'Ahanta West Municipal',
              'Effia Kwesimintsim Municipal',
 'Sekondi Takoradi Metropolitan Area (STMA)',
                             'STMA-Takoradi',
                              'STMA-Sekondi',
 ...
                                   'Wa East',
                              'Wa Municipal',
                             'Nadowli Kaleo',
                      'Daffiama Bussie Issa',
                    'Sissala East Municipal',
                              'Sissala West',
                          'Jirapa Municipal',
                           'Lawra Municipal',
                           'Lambussie Karni',
                                    'Nandom']
Length: 295, dtype: str

In [15]:
population_df["population"].head(20)

0     30832019
1      2060585
2       126576
3       120893
4        94621
5       153140
6       173975
7       245382
8        50065
9        54772
10      140545
11      117224
12       99641
13       52028
14      219109
15      229301
16      179696
17      117999
18      131000
19     2859821
Name: population, dtype: int64

In [16]:
ghana_regions = [
    "Western",
    "Central",
    "Greater Accra",
    "Volta",
    "Eastern",
    "Ashanti",
    "Western North",
    "Ahafo",
    "Bono",
    "Bono East",
    "Oti",
    "Northern",
    "Savannah",
    "North East",
    "Upper East",
    "Upper West"
]

In [17]:
regional_population = population_df[
    population_df["region"].isin(ghana_regions)
].copy()

regional_population

,region,population
1,Western,2060585
19,Central,2859821
44,Greater Accra,5455692
79,Volta,1659040
98,Eastern,2925653
132,Ashanti,5440463
181,Western North,880921
191,Ahafo,564668
198,Bono,1208649
211,Bono East,1203400


In [18]:
submetro_prefixes = (
    "STMA-",
    "CCMA-",
    "AMA-",
    "TMA-",
    "KMA-"
)

district_df = population_df.copy()

district_df["is_region"] = district_df["region"].isin(ghana_regions)

district_df["parent_region"] = (
    district_df["region"]
    .where(district_df["is_region"])
    .ffill()
)

district_df = district_df[
    (district_df["region"] != "Ghana") &
    (~district_df["is_region"])
].copy()

district_df["is_submetro"] = district_df["region"].str.startswith(
    submetro_prefixes,
    na=False
)

district_df = district_df[
    ~district_df["is_submetro"]
].copy()

district_df = district_df[
    ["parent_region", "region", "population"]
].reset_index(drop=True)

district_df.columns = [
    "region",
    "district",
    "population"
]

district_df.head()

,region,district,population
0,Western,Jomoro Municipal,126576
1,Western,Ellembelle,120893
2,Western,Nzema East Municipal,94621
3,Western,Ahanta West Municipal,153140
4,Western,Effia Kwesimintsim Municipal,173975


In [19]:
district_totals = (
    district_df.groupby("region")["population"]
    .sum()
    .reset_index(name="district_population")
)

validation_df = regional_population[
    ["region", "population"]
].merge(
    district_totals,
    on="region",
    how="left"
)

validation_df["difference"] = (
    validation_df["population"] -
    validation_df["district_population"]
)

validation_df

,region,population,district_population,difference
0,Western,2060585,2060585,0
1,Central,2859821,2859821,0
2,Greater Accra,5455692,5455692,0
3,Volta,1659040,1659040,0
4,Eastern,2925653,2925653,0
5,Ashanti,5440463,5440463,0
6,Western North,880921,880921,0
7,Ahafo,564668,564668,0
8,Bono,1208649,1208649,0
9,Bono East,1203400,1203400,0


In [20]:
validation_df["matches"] = validation_df["difference"] == 0

validation_df[["region", "difference", "matches"]]

,region,difference,matches
0,Western,0,True
1,Central,0,True
2,Greater Accra,0,True
3,Volta,0,True
4,Eastern,0,True
5,Ashanti,0,True
6,Western North,0,True
7,Ahafo,0,True
8,Bono,0,True
9,Bono East,0,True


In [16]:
boundary_file = (
    RAW_DATA_DIR /
    "ghana_district_boundaries" /
    "District_261.shp"
)

ghana_gdf = gpd.read_file(boundary_file)

ghana_gdf.head()

,Label,Region,District,geometry
0,Abura Asebu Kwamankese,Central,Abura Asebu Kwamankese,"POLYGON ((-1.1437 5.38852, -1.14374 5.38723, -..."
1,Ada East,Greater Accra,Ada East,"POLYGON ((0.67955 5.77087, 0.67952 5.77089, 0...."
2,Ada West,Greater Accra,Ada West,"POLYGON ((0.47935 6.00641, 0.47936 6.00598, 0...."
3,Agortime-Ziope,Volta,Agortime-Ziope,"POLYGON ((0.64233 6.63246, 0.64243 6.63225, 0...."
4,Adansi Asokwa,Ashanti,Adansi Asokwa,"POLYGON ((-1.43623 6.30419, -1.4357 6.3033, -1..."


In [17]:
print("Dataset dimensions:", ghana_gdf.shape)

print("\nCoordinate Reference System:")
print(ghana_gdf.crs)

print("\nGeometry types:")
print(ghana_gdf.geometry.geom_type.value_counts())

print("\nMissing values:")
print(ghana_gdf.isnull().sum())

Dataset dimensions: (261, 4)

Coordinate Reference System:
EPSG:4326

Geometry types:
Polygon         260
MultiPolygon      1
Name: count, dtype: int64

Missing values:
Label       0
Region      0
District    0
geometry    0
dtype: int64


In [18]:
ghana_gdf["District"].nunique()

261

In [21]:
raster_files = list(RAW_DATA_DIR.rglob(
    "GHA_population_v2_0_gridded_population.tif"
))

if not raster_files:
    raise FileNotFoundError("Population raster was not found.")

raster_path = raster_files[0]



In [22]:
with rasterio.open(raster_path) as src:

    print("Coordinate Reference System:", src.crs)
    print("Dimensions:", src.width, "x", src.height)
    print("Number of bands:", src.count)
    print("Resolution:", src.res)
    print("Bounds:", src.bounds)
    print("Data type:", src.dtypes)
    print("NoData value:", src.nodata)

Coordinate Reference System: EPSG:4326
Dimensions: 5337 x 7721
Number of bands: 1
Resolution: (0.000833333329960652, 0.0008333333300090661)
Bounds: BoundingBox(left=-3.255416639, bottom=4.738750182, right=1.1920833430000002, top=11.172916823)
Data type: ('float32',)
NoData value: -3.3999999521443642e+38


In [26]:
output_file = (
    PROCESSED_DATA_DIR /
    "ghana_district_population_2021.csv"
)

district_df.to_csv(output_file, index=False)

print("Processed dataset saved successfully.")


Processed dataset saved successfully.


In [27]:
processed_df = pd.read_csv(output_file)

print("Dataset dimensions:", processed_df.shape)

processed_df.head()

Dataset dimensions: (261, 3)


,region,district,population
0,Western,Jomoro Municipal,126576
1,Western,Ellembelle,120893
2,Western,Nzema East Municipal,94621
3,Western,Ahanta West Municipal,153140
4,Western,Effia Kwesimintsim Municipal,173975
